In [1]:
!pip install pyspark --quiet
print("PySpark installation complete!")

PySpark installation complete!


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import year, month, to_date, col, round as spark_round

import matplotlib.pyplot as plt
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Creating SparkSession
spark = SparkSession.builder \
        .appName('Day4_Practice') \
        .config('spark.sql.adaptive.enabled', 'true') \
        .getOrCreate()

print(f'Spark version : {spark.version}')
print(f'Spark App Name: {spark.sparkContext.appName}')
print(f'Spark Session : ACTIVE')

Spark version : 4.0.2
Spark App Name: Day4_Practice
Spark Session : ACTIVE


In [4]:
# Load CSV — Bronze Layer
df_bronze = spark.read \
            .option('header', 'true') \
            .option('inferSchema', 'true') \
            .csv('/content/drive/MyDrive/Data_Engineering_Internship/large_sales_data.csv')

print('====BRONZE====')
print(f'Rows: {df_bronze.count()}')
print(f'Columns: {len(df_bronze.columns)}')
print()
df_bronze.printSchema()

====BRONZE====
Rows: 5000
Columns: 13

root
 |-- order_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- revenue: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- sales_rep: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)



In [5]:
# Silver Layer — clean + enrich
df_silver = df_bronze \
            .dropDuplicates() \
            .dropna(subset=['order_id', 'product', 'revenue'])

df_silver = df_silver.withColumn(
    'order_date', to_date(col('order_date'), 'yyyy-MM-dd')
)

df_silver = df_silver \
            .withColumn('order_year',  year(col('order_date'))) \
            .withColumn('order_month', month(col('order_date')))

df_silver = df_silver.withColumn(
    'revenue_category',
    F.when(col('revenue') > 40000, 'High')
     .when(col('revenue') > 10000, 'Medium')
     .otherwise('Low')
)

print(f'Silver layer rows : {df_silver.count()}')
df_silver.select('product', 'revenue', 'order_month', 'order_year', 'revenue_category').show(5, truncate=False)

Silver layer rows : 5000
+--------+-------+-----------+----------+----------------+
|product |revenue|order_month|order_year|revenue_category|
+--------+-------+-----------+----------+----------------+
|Keyboard|13200  |2          |2023      |Medium          |
|Webcam  |17500  |1          |2023      |Medium          |
|Speaker |58500  |4          |2023      |High            |
|Keyboard|9600   |12         |2023      |Low             |
|Laptop  |180000 |8          |2023      |High            |
+--------+-------+-----------+----------+----------------+
only showing top 5 rows


---
## Q1 — Big Data
Explain the 5 Vs of Big Data using an example from the Indian e-commerce industry (like Flipkart).

### The 5 Vs of Big Data

#### 1. Volume
Volume means a very large amount of data being generated every day.
For example, Flipkart receives millions of searches, orders,
clicks, and product views daily.

#### 2. Velocity
Velocity refers to how fast data is generated and processed.
During Big Billion Days sales, thousands of orders are placed every second.

#### 3. Variety
Variety means data comes in different formats.
Companies handle images, videos, customer reviews,
payment records, CSV files, JSON logs, etc.

#### 4. Veracity
Veracity refers to the accuracy and reliability of data.
Data may contain fake reviews, missing values,
duplicate entries, or incorrect addresses.

#### 5. Value
Value means converting raw data into useful insights.
Companies analyze customer behavior to recommend products,
improve user experience, and increase revenue.

## Q2 PySpark Concepts
What is the difference between a Transformation and an Action in PySpark? Give one example of each.

### Transformations vs Actions in PySpark

#### 1. Transformations
Transformations modify or prepare data.
They do not execute immediately.

Examples:
.filter()
.select()
.withColumn()

Spark only stores the instructions.

#### 2. Actions
Actions execute the transformations and produce output.

Examples:
.show()
.count()
.collect()

#### 3. Lazy Evaluation
Spark runs transformations only when an action is called.
This improves performance.

#### 4. Return Values
Transformations return a new DataFrame.

Actions return:
- values
- records
- tables

#### 5. Simple Analogy
Transformations are like writing a recipe.

Actions are like cooking and serving the food.

## Q3 — PySpark Code
Write PySpark code to find the top 5 cities by total revenue from the sales DataFrame.

In [ ]:
# Top 5 cities by total revenue
top5_cities = df_silver \
    .groupBy('city') \
    .agg(F.sum('revenue').alias('total_revenue')) \
    .orderBy(F.desc('total_revenue')) \
    .limit(5)

top5_cities.show(truncate=False)

## Q4 — File Formats
Why is Parquet preferred over CSV for Big Data workloads? Give three specific reasons.

### Why is Parquet Better than CSV?

#### 1. Columnar Storage
Parquet stores data column by column.
If only one column is needed, Spark reads only that column,
making analytics much faster.

#### 2. Compression
Parquet compresses data heavily and reduces file size.

Example:
CSV size = 529 KB
Parquet size = 55 KB

This saves storage and improves performance.

#### 3. Schema Preservation
Parquet stores data types automatically such as:
- integer
- string
- date

CSV stores everything as plain text.

## Q5 — Architecture
Explain the Medallion Architecture. What type of data lives in each layer?

### Medallion Architecture

Medallion Architecture is a data design pattern
that organizes data into 3 layers.

Raw Data -> Bronze -> Silver -> Gold

#### 1. Bronze Layer
This is the raw data layer.
Data is stored exactly as it comes from the source
without cleaning or changes.
#### 2. Silver Layer
This is the cleaned and transformed layer.
Duplicates are removed, null values are handled,
and new columns are added.

#### 3. Gold Layer
This is the business-ready layer.
Data is aggregated and prepared for dashboards and reports.


## Q6 — Architecture
What is the difference between a Data Lake and a Data Warehouse? When would you use each?

### Data Lake vs Data Warehouse

#### 1. Data Lake
A Data Lake stores all types of raw data.

It can store:
- CSV files
- JSON files
- images
- logs
- videos
- Parquet files

The data does not need to be cleaned before storing.

Data Lakes are cheaper and mainly used by:
- Data Engineers
- Data Scientists

Example:
Storing all Flipkart app logs and customer activity data.

#### 2. Data Warehouse
A Data Warehouse stores cleaned and structured data.

The data is organized into tables
so business teams can run queries easily and quickly.

Data Warehouses are faster for analytics
but usually cost more.

They are mainly used by:
- Data Analysts
- Business teams

Example:
Monthly sales reports and business dashboards.

#### 3. Simple Understanding
Data Lake = Store everything first.

Data Warehouse = Store only cleaned and useful data.

#### 4. In Our Project
The Bronze, Silver, and Gold Parquet files
act like a small Data Lake.